In [43]:

import pandas as pd
import transformers
import torch
import numpy as np

merged_df = pd.read_csv("/kaggle/input/datasets/rpghost/merged/merged_dataset.csv")
merged_df

,Unnamed: 0,inchi_key,smiles,hERG_blocker
0,0,ZKLPARSLTMPFCP-UHFFFAOYSA-N,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,0
1,1,AAXGNXOJOSOUQQ-SANMLTNESA-N,CCCCN(Cc1cncn1Cc1ccc(C#N)cc1)[C@H]1CCN(Cc2cccc...,1
2,2,FPDIERBPQFAFSI-UHFFFAOYSA-N,CN1CCN(C2=Nc3ccccc3Nc3sc(CO)cc32)CC1,0
3,3,BRCCXAUQKZKZAB-QHCPKHFHSA-N,N#Cc1ccc(Cn2cncc2CN[C@H]2CCN(c3cccc4ncccc34)C2...,1
4,4,UIEATEWHFDRYRU-UHFFFAOYSA-N,CC(C)COCC(CN(Cc1ccccc1)c1ccccc1)N1CCCC1,1
...,...,...,...,...
10346,10346,ZRJRQVZRIPABBY-UHFFFAOYSA-N,N[C@@H](Cc1cccc(n1)C#N)c1ccccc1-c1noc2cc(ccc12...,1
10347,10347,ZRVYDDFEOLDULP-UHFFFAOYSA-N,FC1(F)CN(CC[C@@H]1CNc1ncccn1)C(=O)OC1Cc2ccccc2C1,0
10348,10348,ZSFYHUCRVPYFMX-UHFFFAOYSA-N,COc1cncc(OC[C@H]2CCNC2)c1-c1cc(Nc2cnc(cn2)C#N)...,0
10349,10349,ZSJAPINLVWEQHA-UHFFFAOYSA-N,N[C@@H](Cc1nc(ccc1F)-c1cn[nH]c1)c1ccccc1-c1noc...,1


In [44]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for file in files:
        print("   ", file)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/rpghost
/kaggle/input/datasets/rpghost/merged
    merged_dataset.csv


In [45]:
import random
import numpy as np
import torch

seed = 123

random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.use_deterministic_algorithms(True)

In [46]:
# from sklearn.model_selection import train_test_split

# X = merged_df["smiles"].values
# y = merged_df["hERG_blocker"].values

# X_trainval, X_test, y_trainval, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.20,
#     random_state=42,
#     stratify=y
# )

# X_train, X_val, y_train, y_val = train_test_split(
#     X_trainval,
#     y_trainval,
#     test_size=0.10,
#     random_state=42,
#     stratify=y_trainval
# )

# print("Train:", len(X_train))
# print("Validation:", len(X_val))
# print("Test:", len(X_test))

In [47]:
!pip install rdkit

In [48]:
# Remove trailing semicolons from SMILES in the original dataframe
merged_df["smiles"] = (
    merged_df["smiles"]
    .astype("string")
    .str.strip()
    .str.replace(r";$", "", regex=True)
)

# Verify the two problematic SMILES
print(
    merged_df[
        merged_df["smiles"].str.contains(
            r"Cc1cc\(cnc1NC\(=O\)|CC1CC1C\(=O\)Nc1ccc",
            regex=True,
            na=False
        )
    ]["smiles"].to_string(index=False)
)

Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)C(=O)N1CC(C)(C)C1)C...
       Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)C#N)C#Cc1ccccc1
Cc1cc(cnc1NC(=O)c1cc(NC(=O)C2COCCO2)ccc1Cl)C#Cc...
Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)C(=O)N1CC(O)C1)C#Cc...
Cc1cc(cnc1NC(=O)c1cc(NC(=O)OC(C)(C)C)ccc1Cl)C#C...
Cc1cc(cnc1NC(=O)c1cc(OCCN2CCOCC2)ccc1Cl)C#Cc1cc...
Cc1cc(cnc1NC(=O)c1cc(NC(=O)OC(C)(C)C)ccc1Cl)C#C...
Cc1cc(cnc1NC(=O)c1cc(NC(=O)OC(C)(C)C)ccc1Cl)C#C...
Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)-c1nc(no1)C1CC1)C#C...
Cc1cc(cnc1NC(=O)c1cc(NC(=O)OC(C)(C)C)ccc1Cl)-c1...
Cc1cc(cnc1NC(=O)c1cc(OCC2COCCO2)ccc1Cl)C#Cc1ccc...
Cc1cc(cnc1NC(=O)c1cc(NC(=O)C2CC(C)(O)C2)ccc1Cl)...
Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)C(=O)N1CC(F)(F)C1)C...
Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)-c1cnn(c1)C1COC1)C#...
Cc1cc(cnc1NC(=O)c1cc(NC(=O)C2COCCO2)ccc1Cl)C#Cc...
Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)-c1noc(n1)C1CCOC1)C...
Cc1cc(cnc1NC(=O)c1cc(ccc1Cl)C(=O)NC(C)(C)CO)C#C...
CC1CC1C(=O)Nc1ccc(Cl)c(c1)C(=O)Nc1ncc(cc1F)C#Cc...
CC1CC1C(=O)Nc1ccc(Cl)c(n1)C(=O)Nc1ncc(cc1C)C#Cc...
Cc1cc(cnc1NC(=O)c1cc(NC(=O)OC(C

In [49]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit

# --------------------------------
# 1. Prepare the dataset
# --------------------------------

df = merged_df.copy()

# Remove rows with missing SMILES or labels
df = df.dropna(subset=["smiles", "hERG_blocker"]).reset_index(drop=True)

# Ensure labels are binary integers
df["hERG_blocker"] = df["hERG_blocker"].astype(int)

# --------------------------------
# 2. Generate Bemis-Murcko scaffolds
# --------------------------------

def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol,
        includeChirality=False
    )

    return scaffold


df["scaffold"] = df["smiles"].apply(get_scaffold)

# Report invalid SMILES rather than silently including them
invalid_count = df["scaffold"].isna().sum()

print("Invalid SMILES:", invalid_count)

df = df.dropna(subset=["scaffold"]).reset_index(drop=True)

# --------------------------------
# 3. Split into train+validation and test
# --------------------------------

groups = df["scaffold"].to_numpy()
indices = np.arange(len(df))

splitter_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=123
)

trainval_idx, test_idx = next(
    splitter_test.split(
        indices,
        groups=groups
    )
)

# --------------------------------
# 4. Split train+validation into train and validation
# --------------------------------

splitter_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=123
)

train_rel_idx, val_rel_idx = next(
    splitter_val.split(
        trainval_idx,
        groups=groups[trainval_idx]
    )
)

train_idx = trainval_idx[train_rel_idx]
val_idx = trainval_idx[val_rel_idx]

# --------------------------------
# 5. Create the dataframes
# --------------------------------

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()
test_df = df.iloc[test_idx].copy()

# Scaffold column is only needed for splitting and validation
train_df.drop(columns=["scaffold"], inplace=True)
val_df.drop(columns=["scaffold"], inplace=True)
test_df.drop(columns=["scaffold"], inplace=True)

# --------------------------------
# 6. Save the files
# --------------------------------

train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)

print("\nScaffold split completed!")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nClass distribution:")
for name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")
    print(split_df["hERG_blocker"].value_counts())
    print(
        split_df["hERG_blocker"]
        .value_counts(normalize=True)
        .round(4)
    )

[11:02:18] SMILES Parse Error: syntax error while parsing: Cc1cc(cnc1NC(=O)c1cc(OCC2COCCO2)ccc1Cl)C#Cc1ccccc1;
[11:02:18] SMILES Parse Error: check for mistakes around position 51:
[11:02:18] 2)ccc1Cl)C#Cc1ccccc1;
[11:02:18] ~~~~~~~~~~~~~~~~~~~~^
[11:02:18] SMILES Parse Error: Failed parsing SMILES 'Cc1cc(cnc1NC(=O)c1cc(OCC2COCCO2)ccc1Cl)C#Cc1ccccc1;' for input: 'Cc1cc(cnc1NC(=O)c1cc(OCC2COCCO2)ccc1Cl)C#Cc1ccccc1;'
[11:02:18] SMILES Parse Error: syntax error while parsing: CC1CC1C(=O)Nc1ccc(Cl)c(c1)C(=O)Nc1ncc(cc1F)C#Cc1ccccc1;
[11:02:18] SMILES Parse Error: check for mistakes around position 55:
[11:02:18] ncc(cc1F)C#Cc1ccccc1;
[11:02:18] ~~~~~~~~~~~~~~~~~~~~^
[11:02:18] SMILES Parse Error: Failed parsing SMILES 'CC1CC1C(=O)Nc1ccc(Cl)c(c1)C(=O)Nc1ncc(cc1F)C#Cc1ccccc1;' for input: 'CC1CC1C(=O)Nc1ccc(Cl)c(c1)C(=O)Nc1ncc(cc1F)C#Cc1ccccc1;'


Invalid SMILES: 2

Scaffold split completed!
Train: 7451
Validation: 954
Test: 1944

Class distribution:

Train:
hERG_blocker
1    4932
0    2519
Name: count, dtype: int64
hERG_blocker
1    0.6619
0    0.3381
Name: proportion, dtype: float64

Validation:
hERG_blocker
1    663
0    291
Name: count, dtype: int64
hERG_blocker
1    0.695
0    0.305
Name: proportion, dtype: float64

Test:
hERG_blocker
1    1329
0     615
Name: count, dtype: int64
hERG_blocker
1    0.6836
0    0.3164
Name: proportion, dtype: float64


In [50]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

def scaffold_set(dataframe):
    return set(
        dataframe["smiles"]
        .apply(
            lambda s: MurckoScaffold.MurckoScaffoldSmiles(
                mol=Chem.MolFromSmiles(s),
                includeChirality=False
            )
        )
    )

train_scaffolds = scaffold_set(train_df)
val_scaffolds = scaffold_set(val_df)
test_scaffolds = scaffold_set(test_df)

print("Train ∩ Validation:",
      len(train_scaffolds & val_scaffolds))

print("Train ∩ Test:",
      len(train_scaffolds & test_scaffolds))

print("Validation ∩ Test:",
      len(val_scaffolds & test_scaffolds))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [51]:
# Extract SMILES and labels from each split

X_train = train_df["smiles"].tolist()
y_train = train_df["hERG_blocker"].tolist()

X_val = val_df["smiles"].tolist()
y_val = val_df["hERG_blocker"].tolist()

X_test = test_df["smiles"].tolist()
y_test = test_df["hERG_blocker"].tolist()

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

Training samples: 7451
Validation samples: 954
Test samples: 1944


In [52]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "DeepChem/ChemBERTa-100M-MLM"

tokenizer = AutoTokenizer.from_pretrained(model_name)
chemberta = AutoModel.from_pretrained(model_name)

# Check token lengths across all three splits
all_smiles = list(X_train) + list(X_val) + list(X_test)

token_lengths = [
    len(tokenizer(smiles, add_special_tokens=True)["input_ids"])
    for smiles in all_smiles
]

print("Maximum token length:", max(token_lengths))
print("Molecules > 512:", sum(length > 512 for length in token_lengths))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

chemberta = chemberta.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-100M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Maximum token length: 436
Molecules > 512: 0
Using: cuda


In [53]:
from torch.utils.data import Dataset


class HERGDataset(Dataset):

    def __init__(self, smiles, labels):

        self.smiles = smiles
        self.labels = labels

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        return (
            self.smiles[idx],
            self.labels[idx]
        )

In [54]:
train_dataset = HERGDataset(
    X_train,
    y_train
)

val_dataset = HERGDataset(
    X_val,
    y_val
)

test_dataset = HERGDataset(
    X_test,
    y_test
)

In [57]:
def collate_fn(batch):

    smiles = [item[0] for item in batch]

    labels = torch.tensor(
    [item[1] for item in batch],
    dtype=torch.float32
    ).unsqueeze(1)

    inputs = tokenizer(
        smiles,
        padding=True,
        truncation=True,
        max_length = 512,
        return_tensors="pt"
    )

    return (
        inputs["input_ids"],
        inputs["attention_mask"],
        labels
    )

In [58]:
from torch.utils.data import DataLoader


g = torch.Generator()
g.manual_seed(seed)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    generator=g,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [59]:
import torch
import torch.nn as nn


class HERG_ChemBERTa(nn.Module):

    def __init__(self):

        super().__init__()

        self.chemberta = AutoModel.from_pretrained(
            "DeepChem/ChemBERTa-100M-MLM"
        )

        hidden_size = self.chemberta.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
        
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
        
            nn.Linear(256, 64),
            nn.ReLU(),
        
            nn.Linear(64, 1)
        )


    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.chemberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden_states = outputs.last_hidden_state

        # [batch, sequence_length, 768]

        mask = attention_mask.unsqueeze(-1)

        # [batch, sequence_length, 1]

        masked_hidden = (
            hidden_states * mask
        )

        summed = masked_hidden.sum(
            dim=1
        )

        counts = mask.sum(
            dim=1
        )

        pooled = summed / counts

        # [batch, 768]

        logits = self.classifier(
            pooled
        )

        return logits

In [60]:
# import torch
# import torch.nn as nn
# from transformers import AutoModel


# class HERG_ChemBERTa_Simple(nn.Module):

#     def __init__(self, dropout=0.1):

#         super().__init__()

#         self.chemberta = AutoModel.from_pretrained(
#             "DeepChem/ChemBERTa-100M-MLM"
#         )

#         hidden_size = self.chemberta.config.hidden_size

#         self.dropout = nn.Dropout(dropout)

#         self.classifier = nn.Linear(
#             hidden_size,
#             1
#         )

#     def forward(self, input_ids, attention_mask):

#         outputs = self.chemberta(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         )

#         hidden_states = outputs.last_hidden_state
#         # Shape: [batch_size, sequence_length, hidden_size]

#         mask = attention_mask.unsqueeze(-1).to(
#             hidden_states.dtype
#         )

#         masked_hidden = hidden_states * mask

#         summed = masked_hidden.sum(dim=1)
#         counts = mask.sum(dim=1).clamp(min=1)

#         pooled = summed / counts
#         # Shape: [batch_size, hidden_size]

#         pooled = self.dropout(pooled)

#         logits = self.classifier(pooled)
#         # Shape: [batch_size, 1]

#         return logits

In [61]:
# simple_classifier = HERG_ChemBERTa_Simple().to(device)

# print(
#     "Total trainable parameters:",
#     sum(
#         p.numel()
#         for p in simple_classifier.parameters()
#         if p.requires_grad
#     )
# )

# print(
#     "Encoder parameters:",
#     sum(
#         p.numel()
#         for p in simple_classifier.chemberta.parameters()
#         if p.requires_grad
#     )
# )

# print(
#     "Classification head parameters:",
#     sum(
#         p.numel()
#         for p in simple_classifier.classifier.parameters()
#         if p.requires_grad
#     )
# )

In [62]:
classifier = HERG_ChemBERTa().to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-100M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [63]:
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in classifier.parameters()
        if p.requires_grad
    )
)

Trainable parameters: 92668545


In [64]:
import numpy as np
import torch

y_train_np = np.asarray(y_train, dtype=int)

n_positive = int((y_train_np == 1).sum())
n_negative = int((y_train_np == 0).sum())

print("Training positives:", n_positive)
print("Training negatives:", n_negative)

assert n_positive > 0 and n_negative > 0, \
    "Both classes must be present in the training split."

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [n_negative / n_positive],
        dtype=torch.float32,
        device=device
    )
)

print("Positive class weight:", n_negative / n_positive)

Training positives: 4932
Training negatives: 2519
Positive class weight: 0.5107461476074615


In [65]:
optimizer = torch.optim.AdamW(
    [
        {
            "params": classifier.chemberta.parameters(),
            "lr": 2e-5
        },
        {
            "params": classifier.classifier.parameters(),
            "lr": 1e-3
        }
    ],
    weight_decay=1e-4
)

In [66]:
print(
    "ChemBERTa trainable parameters:",
    sum(
        p.numel()
        for p in classifier.chemberta.parameters()
        if p.requires_grad
    )
)

print(
    "Classifier trainable parameters:",
    sum(
        p.numel()
        for p in classifier.classifier.parameters()
        if p.requires_grad
    )
)

ChemBERTa trainable parameters: 92126976
Classifier trainable parameters: 541569


In [67]:
input_ids, attention_mask, y_batch = next(
    iter(train_loader)
)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
y_batch = y_batch.to(device)

logits = classifier(
    input_ids,
    attention_mask
)

print("Input IDs:", input_ids.shape)
print("Attention mask:", attention_mask.shape)
print("Labels:", y_batch.shape)
print("Logits:", logits.shape)

Input IDs: torch.Size([32, 70])
Attention mask: torch.Size([32, 70])
Labels: torch.Size([32, 1])
Logits: torch.Size([32, 1])


In [68]:
input_ids, attention_mask, y_batch = next(iter(train_loader))

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
y_batch = y_batch.to(device)

classifier.train()

optimizer.zero_grad()

logits = classifier(input_ids, attention_mask)
loss = criterion(logits, y_batch)

print("Input IDs:", input_ids.shape)
print("Attention mask:", attention_mask.shape)
print("Labels:", y_batch.shape)
print("Logits:", logits.shape)
print("Loss:", loss.item())

assert logits.shape == y_batch.shape
assert torch.isfinite(loss), "Loss is not finite!"

loss.backward()

print("Forward pass and backward pass successful.")
# this is simply a test run to see if everything is working fine or not

Input IDs: torch.Size([32, 154])
Attention mask: torch.Size([32, 154])
Labels: torch.Size([32, 1])
Logits: torch.Size([32, 1])
Loss: 0.5148215889930725
Forward pass and backward pass successful.


In [70]:
from tqdm.auto import tqdm

num_epochs = 50

best_val_loss = float("inf")
patience = 7
patience_counter = 0

for epoch in range(num_epochs):

    # =========================
    # Training
    # =========================

    classifier.train()

    train_loss = 0.0

    for input_ids, attention_mask, y_batch in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1} Training"
    ):

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = classifier(
            input_ids,
            attention_mask
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        train_loss += (
            loss.item() * input_ids.size(0)
        )

    train_loss /= len(train_loader.dataset)


    # =========================
    # Validation
    # =========================

    classifier.eval()

    val_loss = 0.0

    with torch.no_grad():

        for input_ids, attention_mask, y_batch in val_loader:

            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            y_batch = y_batch.to(device)

            logits = classifier(
                input_ids,
                attention_mask
            )

            loss = criterion(
                logits,
                y_batch
            )

            val_loss += (
                loss.item() * input_ids.size(0)
            )

    val_loss /= len(val_loader.dataset)


    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )


    # =========================
    # Check if validation improved
    # =========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        # Save BEST model
        torch.save({
            "epoch": epoch,
            "model_state_dict": classifier.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "patience_counter": patience_counter
        }, "/kaggle/working/best_checkpoint_seed123.pt")

        print("Best model saved.")

    else:

        patience_counter += 1


    # =========================
    # Save LATEST checkpoint
    # =========================

    torch.save({
        "epoch": epoch,
        "model_state_dict": classifier.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "patience_counter": patience_counter
    }, "/kaggle/working/latest_checkpoint_seed123.pt")


    # =========================
    # Early stopping
    # =========================

    if patience_counter >= patience:

        print("Early stopping.")

        break

Epoch 1 Training:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch 01 | Train Loss: 0.4348 | Val Loss: 0.4271
Best model saved.


Epoch 2 Training:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch 02 | Train Loss: 0.3570 | Val Loss: 0.3726
Best model saved.


Epoch 3 Training:   0%|          | 0/233 [00:00<?, ?it/s]

...

Epoch 07 | Train Loss: 0.1605 | Val Loss: 0.5694


Epoch 8 Training:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch 08 | Train Loss: 0.1356 | Val Loss: 0.8110


Epoch 9 Training:   0%|          | 0/233 [00:00<?, ?it/s]

Epoch 09 | Train Loss: 0.1235 | Val Loss: 0.5141
Early stopping.


In [76]:
import numpy as np
import torch

checkpoint = torch.load(
    "/kaggle/working/best_checkpoint_seed123.pt",
    map_location=device,
    weights_only=True
)

classifier.load_state_dict(checkpoint["model_state_dict"])
classifier.eval()

val_probs = []
val_labels = []

with torch.no_grad():
    for input_ids, attention_mask, y_batch in val_loader:

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        logits = classifier(
            input_ids,
            attention_mask
        ).squeeze(-1)

        probs = torch.sigmoid(logits)

        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(y_batch.view(-1).numpy())

val_probs = np.asarray(val_probs)
val_labels = np.asarray(val_labels, dtype=int)

print("Validation probabilities:", val_probs.shape)
print("Validation labels:", val_labels.shape)

Validation probabilities: (954,)
Validation labels: (954,)


In [77]:
from sklearn.metrics import matthews_corrcoef

thresholds = np.arange(0.01, 1.00, 0.01)

threshold_results = []

for threshold in thresholds:

    val_preds = (val_probs >= threshold).astype(int)

    mcc = matthews_corrcoef(val_labels, val_preds)

    threshold_results.append((threshold, mcc))

best_threshold, best_mcc = max(
    threshold_results,
    key=lambda x: x[1]
)

print(f"Best threshold: {best_threshold:.2f}")
print(f"Best validation MCC: {best_mcc:.4f}")

Best threshold: 0.28
Best validation MCC: 0.4121


In [78]:
test_probs = []
test_labels = []

with torch.no_grad():

    for input_ids, attention_mask, y_batch in test_loader:

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        logits = classifier(
            input_ids,
            attention_mask
        ).squeeze(1)

        probs = torch.sigmoid(logits)

        test_probs.extend(
            probs.cpu().numpy()
        )

        test_labels.extend(
            y_batch.numpy()
        )

test_probs = np.array(test_probs)
test_labels = np.array(test_labels)

In [79]:
test_preds = (
    test_probs >= best_threshold
).astype(int)

In [80]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    average_precision_score
)


accuracy = accuracy_score(
    test_labels,
    test_preds
)


precision = precision_score(
    test_labels,
    test_preds,
    zero_division=0
)


recall = recall_score(
    test_labels,
    test_preds,
    zero_division=0
)


f1 = f1_score(
    test_labels,
    test_preds,
    zero_division=0
)


roc_auc = roc_auc_score(
    test_labels,
    test_probs
)


pr_auc = average_precision_score(
    test_labels,
    test_probs
)


mcc = matthews_corrcoef(
    test_labels,
    test_preds
)


cm = confusion_matrix(
    test_labels,
    test_preds
)


print("=" * 50)
print("FINAL TEST RESULTS")
print("=" * 50)

print(f"Threshold : {best_threshold:.2f}")
# print(f"Threshold : {0.5:.2f}")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")
print(f"MCC       : {mcc:.4f}")

print("\nConfusion Matrix:")
print(cm)

FINAL TEST RESULTS
Threshold : 0.28
Accuracy  : 0.7428
Precision : 0.7567
Recall    : 0.9195
F1        : 0.8302
ROC-AUC   : 0.7488
PR-AUC    : 0.8552
MCC       : 0.3479

Confusion Matrix:
[[ 222  393]
 [ 107 1222]]


In [84]:
import zipfile
import os

best_path = "/kaggle/working/best_checkpoint_seed123.pt"
latest_path = "/kaggle/working/latest_checkpoint_seed123.pt"
zip_path = "/kaggle/working/chemberta_checkpoints2.zip"

# Check that both files exist
print("Best checkpoint exists:", os.path.exists(best_path))
print("Latest checkpoint exists:", os.path.exists(latest_path))

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    if os.path.exists(best_path):
        zipf.write(best_path, "best_checkpoint_seed123.pt")
    else:
        print("Best checkpoint not found!")

    if os.path.exists(latest_path):
        zipf.write(latest_path, "latest_checkpoint_seed123.pt")
    else:
        print("Latest checkpoint not found!")

print("ZIP created:", zip_path)

Best checkpoint exists: True
Latest checkpoint exists: True
ZIP created: /kaggle/working/chemberta_checkpoints2.zip
